# ML Experiment Notebook

This notebook runs the end-to-end experiment for your project:

- Generate sample image-caption data
- Train CNN + LSTM caption model
- Test caption generation on one image
- Keep commands ready for Flask/Streamlit app launch


In [ ]:
# 1) Imports and environment checks
import os
import sys
import json
import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Working directory:', os.getcwd())

In [ ]:
# 2) Ensure project root as current folder
# Run this notebook from ml/ folder.
PROJECT_ROOT = os.getcwd()
print('PROJECT_ROOT =', PROJECT_ROOT)

required_paths = [
    'src/create_sample_dataset.py',
    'src/train_caption.py',
    'src/models/caption_model.py',
    'src/utils/text.py'
]

for p in required_paths:
    print(p, '->', 'OK' if os.path.exists(p) else 'MISSING')

In [ ]:
# 3) Generate sample dataset
# Creates data/sample_images and data/captions.csv
!python src/create_sample_dataset.py --samples 120

In [ ]:
# 4) Train caption model
# Artifacts will be stored in artifacts/
!python src/train_caption.py --csv data/captions.csv --epochs 5 --batch-size 16

In [ ]:
# 5) Quick single-image caption test
import cv2
from torchvision import transforms

sys.path.append('src')
from models.caption_model import CaptionNet
from utils.text import Vocabulary


def frame_to_tensor(frame_bgr, device):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    return transform(rgb).unsqueeze(0).to(device)


def predict_caption(image_path, weights='artifacts/caption_model.pt', vocab_path='artifacts/vocab.json'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    vocab = Vocabulary.load(vocab_path)
    ckpt = torch.load(weights, map_location=device)

    model = CaptionNet(
        embed_size=ckpt['embed_size'],
        hidden_size=ckpt['hidden_size'],
        vocab_size=ckpt['vocab_size'],
    ).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()

    frame = cv2.imread(image_path)
    if frame is None:
        raise ValueError(f'Could not read image: {image_path}')

    with torch.no_grad():
        image = frame_to_tensor(frame, device)
        features = model.encoder(image)
        ids = model.decoder.sample(
            features=features,
            max_len=ckpt.get('max_len', 25),
            start_id=vocab.stoi['<start>'],
            end_id=vocab.stoi['<end>'],
        )
    return vocab.decode(ids)

sample_image = 'data/sample_images/sample_000.png'
if os.path.exists(sample_image):
    print('Caption:', predict_caption(sample_image))
else:
    print('Sample image not found. Run dataset generation first.')

## Run Web Apps

After training, run either app from terminal:

```bash
python flask_app.py --weights artifacts/caption_model.pt --vocab artifacts/vocab.json
```

or

```bash
python -m streamlit run streamlit_app.py
```


In [ ]:
# 6) AWS SDK check (boto3)
import boto3

print('boto3 version:', boto3.__version__)

# Optional: verify session/region
session = boto3.Session()
print('AWS profile:', session.profile_name)
print('AWS region:', session.region_name)
